# Исследование плотности независимых ценовых движений

Ноутбук проверяет гипотезу: существует ли временной горизонт, на котором плотность независимых значимых движений максимальна, и насколько этот горизонт устойчив для разных криптовалют, порогов движения и рыночных периодов.

Источник данных: `s3://binance-data-downloader/minute_returns_dataset/features/minute_returns_1m_2020-02-01_2026-02-01.parquet`.

## Метод

Для каждой валюты, горизонта `h = 1..120` минут и порога `0.5%`, `1.0%`, `1.5%`:

1. считаем будущую составную доходность за следующие `h` минут;
2. отмечаем движения, где `abs(return_h) >= threshold`;
3. оставляем только независимые неперекрывающиеся события жадным проходом слева направо;
4. считаем плотность как число независимых событий в день доступных наблюдений.

Так короткие горизонты не получают искусственное преимущество из-за множества почти одинаковых перекрывающихся сигналов.

In [ ]:
from __future__ import annotations

import io
import math
import os
import sys
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "build_price_feature_day.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from build_price_feature_day import make_s3_client

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)
plt.style.use("seaborn-v0_8-whitegrid")

## Конфигурация

`SELECTED_SYMBOLS = None` означает использовать все колонки доходностей. Для быстрого прогона можно указать, например, `['BTCUSDT', 'ETHUSDT', 'SOLUSDT']`.

`RUN_BOOTSTRAP` по умолчанию выключен, потому что блочный бутстреп пересчитывает много кривых плотности.

In [ ]:
S3_BUCKET = "binance-data-downloader"
S3_KEY = "minute_returns_dataset/features/minute_returns_1m_2020-02-01_2026-02-01.parquet"

CACHE_DIR = PROJECT_ROOT / "analysis" / "cache"
LOCAL_PARQUET = CACHE_DIR / Path(S3_KEY).name

SELECTED_SYMBOLS: list[str] | None = None
HORIZONS = list(range(1, 121))
THRESHOLDS = [0.005, 0.010, 0.015]
PLATEAU_LEVEL = 0.95
STABILITY_FREQ = "QE"  # monthly: 'ME', quarterly: 'QE', rolling handled below separately

RUN_BOOTSTRAP = False
BOOTSTRAP_ITERATIONS = 300
BOOTSTRAP_RANDOM_SEED = 42

## Загрузка parquet из S3

Файл кешируется локально, чтобы последующие запуски не скачивали весь parquet заново. Чтение колонок после этого идет через `pyarrow`.

In [ ]:
def ensure_local_parquet(bucket: str = S3_BUCKET, key: str = S3_KEY, local_path: Path = LOCAL_PARQUET) -> Path:
    local_path.parent.mkdir(parents=True, exist_ok=True)
    if local_path.exists() and local_path.stat().st_size > 0:
        print(f"Using cached parquet: {local_path} ({local_path.stat().st_size / 1024**2:.1f} MiB)")
        return local_path

    print(f"Downloading s3://{bucket}/{key} -> {local_path}")
    s3 = make_s3_client()
    s3.download_file(bucket, key, str(local_path))
    print(f"Downloaded {local_path.stat().st_size / 1024**2:.1f} MiB")
    return local_path


def parquet_columns(path: Path) -> list[str]:
    return pq.ParquetFile(path).schema.names


def symbol_from_return_column(column: str) -> str:
    return column.removesuffix("_return")


def return_column_for_symbol(symbol: str) -> str:
    return symbol if symbol.endswith("_return") else f"{symbol}_return"


local_parquet = ensure_local_parquet()
all_columns = parquet_columns(local_parquet)
timestamp_column = "timestamp"
return_columns = [c for c in all_columns if c != timestamp_column and c.endswith("_return")]

if SELECTED_SYMBOLS is None:
    selected_return_columns = return_columns
else:
    selected_return_columns = [return_column_for_symbol(s) for s in SELECTED_SYMBOLS]
    missing = sorted(set(selected_return_columns) - set(return_columns))
    if missing:
        raise ValueError(f"Missing return columns in parquet: {missing}")

read_columns = [timestamp_column, *selected_return_columns]
returns = pd.read_parquet(local_parquet, columns=read_columns)
returns[timestamp_column] = pd.to_datetime(returns[timestamp_column], utc=True)
returns = returns.sort_values(timestamp_column).drop_duplicates(timestamp_column).reset_index(drop=True)

print(f"Rows: {len(returns):,}")
print(f"Date range: {returns[timestamp_column].min()} .. {returns[timestamp_column].max()}")
print(f"Symbols: {len(selected_return_columns)}")
display(returns.head())

## Аудит покрытия и пропусков

Перед расчетом плотности важно отделить поздний старт инструмента от внутренних пропусков. `missing_before_first` показывает минуты до первого валидного return, а `internal_missing` — пропуски внутри активного периода валюты.

In [ ]:
def missingness_report(frame: pd.DataFrame, return_cols: list[str]) -> tuple[pd.DataFrame, pd.DataFrame]:
    rows = []
    gap_rows = []
    total = len(frame)

    for col in return_cols:
        symbol = symbol_from_return_column(col)
        valid = frame[col].notna().to_numpy()
        non_null = int(valid.sum())

        if non_null == 0:
            rows.append(
                {
                    "symbol": symbol,
                    "total_rows": total,
                    "non_null": 0,
                    "null_total": total,
                    "coverage_pct": 0.0,
                    "first_valid": pd.NaT,
                    "last_valid": pd.NaT,
                    "missing_before_first": total,
                    "missing_after_last": 0,
                    "internal_missing": 0,
                    "active_span_minutes": 0,
                    "active_coverage_pct": np.nan,
                }
            )
            continue

        valid_idx = np.flatnonzero(valid)
        first_idx = int(valid_idx[0])
        last_idx = int(valid_idx[-1])
        active = frame[col].iloc[first_idx : last_idx + 1]
        internal_mask = active.isna().to_numpy()
        internal_missing = int(internal_mask.sum())
        active_span = last_idx - first_idx + 1

        rows.append(
            {
                "symbol": symbol,
                "total_rows": total,
                "non_null": non_null,
                "null_total": total - non_null,
                "coverage_pct": non_null / total * 100,
                "first_valid": frame.loc[first_idx, timestamp_column],
                "last_valid": frame.loc[last_idx, timestamp_column],
                "missing_before_first": first_idx,
                "missing_after_last": total - last_idx - 1,
                "internal_missing": internal_missing,
                "active_span_minutes": active_span,
                "active_coverage_pct": non_null / active_span * 100,
            }
        )

        if internal_missing:
            starts = np.flatnonzero(internal_mask & np.r_[True, ~internal_mask[:-1]])
            ends = np.flatnonzero(internal_mask & np.r_[~internal_mask[1:], True])
            for start, end in zip(starts, ends):
                abs_start = first_idx + int(start)
                abs_end = first_idx + int(end)
                gap_rows.append(
                    {
                        "symbol": symbol,
                        "gap_start": frame.loc[abs_start, timestamp_column],
                        "gap_end": frame.loc[abs_end, timestamp_column],
                        "missing_minutes": int(end - start + 1),
                    }
                )

    summary = pd.DataFrame(rows).sort_values(["first_valid", "symbol"], na_position="last")
    gaps = pd.DataFrame(gap_rows).sort_values(["missing_minutes", "gap_start"], ascending=[False, True])
    return summary, gaps


missingness, internal_gaps = missingness_report(returns, selected_return_columns)
display(missingness)

if internal_gaps.empty:
    print("No internal gaps inside active symbol spans.")
else:
    gap_summary = (
        internal_gaps.groupby("symbol")
        .agg(
            gap_count=("missing_minutes", "size"),
            total_internal_missing=("missing_minutes", "sum"),
            max_gap_minutes=("missing_minutes", "max"),
        )
        .sort_values(["total_internal_missing", "max_gap_minutes"], ascending=False)
    )
    display(gap_summary)
    display(internal_gaps.head(50))

missingness.to_csv(CACHE_DIR / "minute_returns_missingness_summary.csv", index=False)
internal_gaps.to_csv(CACHE_DIR / "minute_returns_internal_gaps.csv", index=False)

## Реализация метрики плотности

In [ ]:
@dataclass(frozen=True)
class DensityPoint:
    symbol: str
    threshold: float
    horizon: int
    independent_events: int
    possible_observations: int
    density_per_day: float
    event_share: float


def forward_compound_return(minute_returns: pd.Series, horizon: int) -> pd.Series:
    """Return from t+1 through t+h, using log1p for numerical stability."""
    values = minute_returns.astype("float64").replace([np.inf, -np.inf], np.nan)
    log_values = np.log1p(values)
    rolled = log_values.shift(-1).rolling(window=horizon, min_periods=horizon).sum().shift(-(horizon - 1))
    return np.expm1(rolled)


def count_non_overlapping_events(mask: np.ndarray, horizon: int) -> int:
    starts = np.flatnonzero(mask)
    count = 0
    blocked_until = -1
    for start in starts:
        if start >= blocked_until:
            count += 1
            blocked_until = start + horizon
    return count


def density_curve_for_series(
    minute_returns: pd.Series,
    symbol: str,
    horizons: list[int] = HORIZONS,
    thresholds: list[float] = THRESHOLDS,
) -> pd.DataFrame:
    points: list[DensityPoint] = []
    for h in horizons:
        future = forward_compound_return(minute_returns, h)
        possible = int(future.notna().sum())
        possible_days = possible / 1440.0
        abs_future = future.abs().to_numpy()
        finite = np.isfinite(abs_future)

        for threshold in thresholds:
            mask = finite & (abs_future >= threshold)
            events = count_non_overlapping_events(mask, h)
            points.append(
                DensityPoint(
                    symbol=symbol,
                    threshold=threshold,
                    horizon=h,
                    independent_events=events,
                    possible_observations=possible,
                    density_per_day=events / possible_days if possible_days else np.nan,
                    event_share=events / possible if possible else np.nan,
                )
            )
    return pd.DataFrame(points)


def build_density_curves(frame: pd.DataFrame, return_cols: list[str]) -> pd.DataFrame:
    curves = []
    for col in return_cols:
        symbol = symbol_from_return_column(col)
        print(f"Computing density curves for {symbol}...")
        curves.append(density_curve_for_series(frame[col], symbol=symbol))
    return pd.concat(curves, ignore_index=True)


density = build_density_curves(returns, selected_return_columns)
display(density.head())

## Первичный анализ формы

Строим кривые `D(h)` и считаем максимум, плато почти оптимальных горизонтов и выраженность максимума.

In [ ]:
def summarize_curve(group: pd.DataFrame, plateau_level: float = PLATEAU_LEVEL) -> pd.Series:
    ordered = group.sort_values("horizon")
    valid_density = ordered["density_per_day"].dropna()
    if valid_density.empty:
        return pd.Series(
            {
                "best_horizon": np.nan,
                "max_density_per_day": np.nan,
                "mean_density_per_day": np.nan,
                "max_to_mean": np.nan,
                "neighbor_advantage": np.nan,
                "plateau_min_h": np.nan,
                "plateau_max_h": np.nan,
                "plateau_width": np.nan,
            }
        )
    idx = valid_density.idxmax()
    best = ordered.loc[idx]
    max_density = float(best["density_per_day"])
    plateau = ordered.loc[ordered["density_per_day"] >= plateau_level * max_density, "horizon"]
    neighbor = ordered.loc[ordered["horizon"].between(best["horizon"] - 2, best["horizon"] + 2)]
    neighbor_without_best = neighbor.loc[neighbor["horizon"] != best["horizon"]]
    neighbor_advantage = (
        max_density / neighbor_without_best["density_per_day"].mean() - 1.0
        if len(neighbor_without_best) and neighbor_without_best["density_per_day"].mean() > 0
        else np.nan
    )
    return pd.Series(
        {
            "best_horizon": int(best["horizon"]),
            "max_density_per_day": max_density,
            "mean_density_per_day": float(ordered["density_per_day"].mean()),
            "max_to_mean": max_density / valid_density.mean(),
            "neighbor_advantage": neighbor_advantage,
            "plateau_min_h": int(plateau.min()),
            "plateau_max_h": int(plateau.max()),
            "plateau_width": int(plateau.max() - plateau.min() + 1),
        }
    )


def summarize_curves_by(frame: pd.DataFrame, keys: list[str]) -> pd.DataFrame:
    rows = []
    for key_values, group in frame.groupby(keys, dropna=False):
        if not isinstance(key_values, tuple):
            key_values = (key_values,)
        row = dict(zip(keys, key_values))
        row.update(summarize_curve(group).to_dict())
        rows.append(row)
    return pd.DataFrame(rows)


summary = summarize_curves_by(density, ["symbol", "threshold"])
summary["threshold_pct"] = summary["threshold"].mul(100)
display(summary.sort_values(["threshold", "best_horizon", "symbol"]))

In [ ]:
def plot_density_curves(curves: pd.DataFrame, symbols: list[str] | None = None) -> None:
    symbols = symbols or sorted(curves["symbol"].unique())
    thresholds = sorted(curves["threshold"].unique())
    fig, axes = plt.subplots(len(thresholds), 1, figsize=(12, 4 * len(thresholds)), sharex=True)
    if len(thresholds) == 1:
        axes = [axes]

    for ax, threshold in zip(axes, thresholds):
        subset = curves[(curves["threshold"] == threshold) & (curves["symbol"].isin(symbols))]
        for symbol, part in subset.groupby("symbol"):
            ax.plot(part["horizon"], part["density_per_day"], label=symbol, linewidth=1.8)
        ax.set_title(f"Плотность независимых движений, threshold={threshold:.1%}")
        ax.set_ylabel("events per day")
        ax.legend(ncol=4, fontsize=9)
    axes[-1].set_xlabel("horizon, minutes")
    plt.tight_layout()


plot_density_curves(density)

## Сравнение валют и порогов

Смотрим не только точку максимума, но и диапазон почти оптимальных горизонтов.

In [ ]:
best_horizon_table = summary.pivot(index="symbol", columns="threshold_pct", values="best_horizon").sort_index()
plateau_table = summary.assign(
    plateau=lambda x: x["plateau_min_h"].astype(str) + "-" + x["plateau_max_h"].astype(str)
).pivot(index="symbol", columns="threshold_pct", values="plateau").sort_index()
density_table = summary.pivot(index="symbol", columns="threshold_pct", values="max_density_per_day").sort_index()

print("Best horizon, minutes")
display(best_horizon_table)
print("95% plateau, minutes")
display(plateau_table)
print("Max density, independent events per day")
display(density_table)

In [ ]:
def plot_heatmap(table: pd.DataFrame, title: str, fmt: str = ".0f") -> None:
    values = table.to_numpy(dtype=float)
    fig, ax = plt.subplots(figsize=(1.4 * len(table.columns) + 5, 0.35 * len(table.index) + 3))
    im = ax.imshow(values, aspect="auto", cmap="viridis")
    ax.set_title(title)
    ax.set_xticks(range(len(table.columns)), [f"{c:.1f}%" for c in table.columns])
    ax.set_yticks(range(len(table.index)), table.index)
    for i in range(values.shape[0]):
        for j in range(values.shape[1]):
            if np.isfinite(values[i, j]):
                ax.text(j, i, format(values[i, j], fmt), ha="center", va="center", color="white", fontsize=9)
    fig.colorbar(im, ax=ax, shrink=0.8)
    plt.tight_layout()


plot_heatmap(best_horizon_table, "Оптимальный горизонт по валютам и порогам", fmt=".0f")
plot_heatmap(density_table, "Максимальная плотность, событий в день", fmt=".2f")

## Устойчивость во времени

Разбиваем историю на календарные периоды и заново считаем оптимальные горизонты. Если максимум стабилен, распределение `best_horizon` будет узким.

In [ ]:
def normalize_pandas_freq(freq: str) -> str:
    # pandas 3 removed legacy aliases such as 'M' and 'Q'.
    return {"M": "ME", "Q": "QE"}.get(freq, freq)


def period_density_curves(
    frame: pd.DataFrame,
    return_cols: list[str],
    freq: str = STABILITY_FREQ,
) -> pd.DataFrame:
    result = []
    periods = frame.set_index(timestamp_column).groupby(pd.Grouper(freq=normalize_pandas_freq(freq)))
    for period_end, period_frame in periods:
        if period_frame.empty:
            continue
        period_frame = period_frame.reset_index()
        period_start = period_frame[timestamp_column].min()
        period_label = f"{period_start.date()}..{period_frame[timestamp_column].max().date()}"
        print(f"Period {period_label}: rows={len(period_frame):,}")
        curves = build_density_curves(period_frame, return_cols)
        curves["period_start"] = period_start
        curves["period_end"] = period_frame[timestamp_column].max()
        curves["period_label"] = period_label
        result.append(curves)
    return pd.concat(result, ignore_index=True) if result else pd.DataFrame()


period_density = period_density_curves(returns, selected_return_columns)
period_summary = summarize_curves_by(
    period_density,
    ["period_label", "period_start", "period_end", "symbol", "threshold"],
)
period_summary["threshold_pct"] = period_summary["threshold"].mul(100)
display(period_summary.head())

In [ ]:
stability = (
    period_summary.groupby(["symbol", "threshold_pct"])
    .agg(
        periods=("best_horizon", "size"),
        mean_best_h=("best_horizon", "mean"),
        std_best_h=("best_horizon", "std"),
        min_best_h=("best_horizon", "min"),
        max_best_h=("best_horizon", "max"),
        median_plateau_width=("plateau_width", "median"),
        mean_max_density=("max_density_per_day", "mean"),
    )
    .reset_index()
)
stability["cv_best_h"] = stability["std_best_h"] / stability["mean_best_h"]
display(stability.sort_values(["threshold_pct", "cv_best_h", "symbol"]))

In [ ]:
def plot_period_best_horizons(period_stats: pd.DataFrame) -> None:
    thresholds = sorted(period_stats["threshold_pct"].unique())
    fig, axes = plt.subplots(len(thresholds), 1, figsize=(13, 4 * len(thresholds)), sharex=True)
    if len(thresholds) == 1:
        axes = [axes]

    for ax, threshold_pct in zip(axes, thresholds):
        subset = period_stats[period_stats["threshold_pct"] == threshold_pct]
        for symbol, part in subset.groupby("symbol"):
            ax.plot(part["period_start"], part["best_horizon"], marker="o", linewidth=1.4, label=symbol)
        ax.set_title(f"Устойчивость оптимального горизонта, threshold={threshold_pct:.1f}%")
        ax.set_ylabel("best horizon, min")
        ax.legend(ncol=4, fontsize=9)
    axes[-1].set_xlabel("period")
    plt.tight_layout()


plot_period_best_horizons(period_summary)

## Месячная динамика 95%-диапазона

Здесь для каждого месяца, валюты и порога считается диапазон горизонтов, на котором плотность не ниже `95%` от месячного максимума. Важны три величины: левая граница `plateau_min_h`, правая граница `plateau_max_h` и ширина `plateau_width`.

In [ ]:
MONTHLY_PLATEAU_FREQ = "ME"
MONTHLY_PLATEAU_SYMBOLS: list[str] | None = None  # e.g. ["BTCUSDT", "ETHUSDT", "XRPUSDT"]

monthly_density = period_density_curves(
    returns,
    selected_return_columns,
    freq=MONTHLY_PLATEAU_FREQ,
)
monthly_plateau = summarize_curves_by(
    monthly_density,
    ["period_label", "period_start", "period_end", "symbol", "threshold"],
)
monthly_plateau["threshold_pct"] = monthly_plateau["threshold"].mul(100)
monthly_plateau["month"] = monthly_plateau["period_start"].dt.to_period("M").astype(str)

monthly_plateau_path = CACHE_DIR / "monthly_95_plateau_summary.csv"
monthly_plateau.to_csv(monthly_plateau_path, index=False)
print(f"Saved {monthly_plateau_path}")
display(
    monthly_plateau[
        [
            "month",
            "symbol",
            "threshold_pct",
            "best_horizon",
            "plateau_min_h",
            "plateau_max_h",
            "plateau_width",
            "max_density_per_day",
        ]
    ].sort_values(["symbol", "threshold_pct", "month"])
)

In [ ]:
def plot_monthly_plateau_bands(
    monthly_stats: pd.DataFrame,
    symbols: list[str] | None = MONTHLY_PLATEAU_SYMBOLS,
) -> None:
    if symbols is None:
        symbols = sorted(monthly_stats["symbol"].dropna().unique())
    thresholds = sorted(monthly_stats["threshold_pct"].dropna().unique())

    for symbol in symbols:
        fig, axes = plt.subplots(len(thresholds), 1, figsize=(14, 3.5 * len(thresholds)), sharex=True)
        if len(thresholds) == 1:
            axes = [axes]

        for ax, threshold_pct in zip(axes, thresholds):
            part = monthly_stats[
                (monthly_stats["symbol"] == symbol)
                & (monthly_stats["threshold_pct"] == threshold_pct)
            ].dropna(subset=["plateau_min_h", "plateau_max_h", "best_horizon"])
            part = part.sort_values("period_start")
            if part.empty:
                ax.set_title(f"{symbol}, threshold={threshold_pct:.1f}%: no data")
                continue

            x = part["period_start"]
            y_min = part["plateau_min_h"].astype(float)
            y_max = part["plateau_max_h"].astype(float)
            y_best = part["best_horizon"].astype(float)
            ax.fill_between(x, y_min, y_max, alpha=0.22, label="95% plateau")
            ax.plot(x, y_best, marker="o", linewidth=1.5, label="best horizon")
            ax.set_title(f"{symbol}, threshold={threshold_pct:.1f}%")
            ax.set_ylabel("horizon, min")
            ax.legend(loc="upper left")

        axes[-1].set_xlabel("month")
        fig.suptitle(f"Monthly 95% density plateau: {symbol}", y=1.01, fontsize=14)
        plt.tight_layout()


plot_monthly_plateau_bands(monthly_plateau)

In [ ]:
def plot_monthly_plateau_width(
    monthly_stats: pd.DataFrame,
    symbols: list[str] | None = MONTHLY_PLATEAU_SYMBOLS,
) -> None:
    if symbols is None:
        symbols = sorted(monthly_stats["symbol"].dropna().unique())
    thresholds = sorted(monthly_stats["threshold_pct"].dropna().unique())
    fig, axes = plt.subplots(len(thresholds), 1, figsize=(14, 3.8 * len(thresholds)), sharex=True)
    if len(thresholds) == 1:
        axes = [axes]

    for ax, threshold_pct in zip(axes, thresholds):
        subset = monthly_stats[
            (monthly_stats["threshold_pct"] == threshold_pct)
            & (monthly_stats["symbol"].isin(symbols))
        ].dropna(subset=["plateau_width"])
        for symbol, part in subset.groupby("symbol"):
            part = part.sort_values("period_start")
            ax.plot(part["period_start"], part["plateau_width"], marker="o", linewidth=1.3, label=symbol)
        ax.set_title(f"Monthly width of 95% plateau, threshold={threshold_pct:.1f}%")
        ax.set_ylabel("width, min")
        ax.legend(ncol=4, fontsize=9)
    axes[-1].set_xlabel("month")
    plt.tight_layout()


plot_monthly_plateau_width(monthly_plateau)

In [ ]:
def plot_plateau_band_by_symbol(
    stats: pd.DataFrame,
    symbol: str,
    period_label: str = "month",
    title_prefix: str | None = None,
    best_col: str = "best_horizon",
    min_col: str = "plateau_min_h",
    max_col: str = "plateau_max_h",
    threshold_col: str = "threshold_pct",
    date_col: str = "period_start",
) -> None:
    part_all = stats.loc[stats["symbol"] == symbol].copy()
    if part_all.empty:
        raise ValueError(f"No rows for symbol={symbol}")

    thresholds = sorted(part_all[threshold_col].dropna().unique())
    fig, axes = plt.subplots(
        len(thresholds),
        1,
        figsize=(16, 3.6 * len(thresholds)),
        sharex=True,
    )
    if len(thresholds) == 1:
        axes = [axes]

    for ax, threshold_pct in zip(axes, thresholds):
        part = part_all.loc[part_all[threshold_col] == threshold_pct].dropna(
            subset=[date_col, min_col, max_col, best_col]
        )
        part = part.sort_values(date_col)
        if part.empty:
            ax.set_title(f"{symbol}, threshold={threshold_pct:.1f}%: no data")
            continue

        x = pd.to_datetime(part[date_col])
        y_min = part[min_col].astype(float)
        y_max = part[max_col].astype(float)
        y_best = part[best_col].astype(float)

        ax.fill_between(x, y_min, y_max, alpha=0.22, label="95% plateau")
        ax.plot(x, y_best, marker="o", linewidth=1.7, label="best horizon")
        ax.set_title(f"{symbol}, threshold={threshold_pct:.1f}%")
        ax.set_ylabel("horizon, min")
        ax.grid(True, alpha=0.65)
        ax.legend(loc="upper left")

    axes[-1].set_xlabel(period_label)
    title_prefix = title_prefix or f"{period_label.capitalize()}ly 95% density plateau"
    fig.suptitle(f"{title_prefix}: {symbol}", y=1.01, fontsize=14)
    plt.tight_layout()


plot_plateau_band_by_symbol(monthly_plateau, "BCHUSDT", period_label="month")

## Нормированные кривые и универсальная форма

Нормируем каждую кривую на собственный максимум: `D(h) / Dmax`. Если формы похожи, медианная кривая и межквартильный интервал должны показывать общий пик и последующий спад.

In [ ]:
normalized = density.copy()
normalized["density_norm"] = normalized.groupby(["symbol", "threshold"])["density_per_day"].transform(lambda s: s / s.max())

norm_summary = (
    normalized.groupby(["threshold", "horizon"])["density_norm"]
    .agg(q25=lambda s: s.quantile(0.25), median="median", q75=lambda s: s.quantile(0.75), mean="mean")
    .reset_index()
)

fig, axes = plt.subplots(len(THRESHOLDS), 1, figsize=(12, 4 * len(THRESHOLDS)), sharex=True)
if len(THRESHOLDS) == 1:
    axes = [axes]
for ax, threshold in zip(axes, THRESHOLDS):
    part = norm_summary[norm_summary["threshold"] == threshold]
    ax.plot(part["horizon"], part["median"], label="median", color="black", linewidth=2.2)
    ax.fill_between(part["horizon"], part["q25"], part["q75"], alpha=0.25, label="IQR")
    ax.set_title(f"Нормированная форма D(h)/Dmax, threshold={threshold:.1%}")
    ax.set_ylabel("normalized density")
    ax.legend()
axes[-1].set_xlabel("horizon, minutes")
plt.tight_layout()

## Бутстреп устойчивости максимума

Опциональный блок: ресемплирует дни с возвращением и пересчитывает оптимальный горизонт. Это дает доверительный интервал положения максимума. Для полного датасета блок может выполняться долго.

In [ ]:
def bootstrap_best_horizons(
    frame: pd.DataFrame,
    return_col: str,
    threshold: float,
    iterations: int = BOOTSTRAP_ITERATIONS,
    seed: int = BOOTSTRAP_RANDOM_SEED,
) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    symbol = symbol_from_return_column(return_col)
    tmp = frame[[timestamp_column, return_col]].copy()
    tmp["date"] = tmp[timestamp_column].dt.floor("D")
    days = [day_frame[return_col].reset_index(drop=True) for _, day_frame in tmp.groupby("date") if len(day_frame) > max(HORIZONS)]
    if not days:
        return pd.DataFrame()

    rows = []
    for i in range(iterations):
        sampled = rng.choice(len(days), size=len(days), replace=True)
        series = pd.concat([days[j] for j in sampled], ignore_index=True)
        curve = density_curve_for_series(series, symbol=symbol, thresholds=[threshold])
        best = summarize_curve(curve)
        rows.append(
            {
                "symbol": symbol,
                "threshold": threshold,
                "iteration": i,
                "best_horizon": best["best_horizon"],
                "max_density_per_day": best["max_density_per_day"],
            }
        )
    return pd.DataFrame(rows)


if RUN_BOOTSTRAP:
    boot_parts = []
    for col in selected_return_columns:
        for threshold in THRESHOLDS:
            print(f"Bootstrap {symbol_from_return_column(col)}, threshold={threshold:.1%}")
            boot_parts.append(bootstrap_best_horizons(returns, col, threshold))
    bootstrap = pd.concat(boot_parts, ignore_index=True)
    bootstrap_summary = (
        bootstrap.groupby(["symbol", "threshold"])["best_horizon"]
        .agg(mean="mean", std="std", q025=lambda s: s.quantile(0.025), q50="median", q975=lambda s: s.quantile(0.975))
        .reset_index()
    )
    bootstrap_summary["threshold_pct"] = bootstrap_summary["threshold"].mul(100)
    display(bootstrap_summary)
else:
    print("RUN_BOOTSTRAP=False; set it to True to run block bootstrap.")

## Итоговые выводы

Заполните после выполнения ноутбука:

- есть ли выраженный максимум `D(h)` для большинства валют;
- где находится типичный оптимальный диапазон для каждого порога;
- смещается ли максимум вправо при росте порога движения;
- насколько стабилен оптимальный горизонт по кварталам/месяцам;
- похожи ли нормированные формы кривых между валютами;
- какие валюты выбиваются из общей закономерности.